In [32]:
from datasets import load_dataset
dataset=load_dataset('conll2003',trust_remote_code=True,download_mode='force_redownload')

Generating test split: 100%|██████████| 3453/3453 [00:00<00:00, 9023.74 examples/s]


In [33]:
dataset

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3453
    })
})

In [34]:
dataset['train'][0]

{'id': '0',
 'tokens': ['EU',
  'rejects',
  'German',
  'call',
  'to',
  'boycott',
  'British',
  'lamb',
  '.'],
 'pos_tags': [22, 42, 16, 21, 35, 37, 16, 21, 7],
 'chunk_tags': [11, 21, 11, 12, 21, 22, 11, 12, 0],
 'ner_tags': [3, 0, 7, 0, 0, 0, 7, 0, 0]}

In [35]:
tagnames=dataset['train'].features['ner_tags'].feature.names

In [36]:
len(tagnames)

9

In [37]:
from collections import Counter
counter=Counter()
for example in dataset['train']:
    counter.update(token.lower() for token in example['tokens'])
vocab={'PAD':0,'UNK':1}
for word in counter:
    vocab[word]=len(vocab)

In [38]:
len(vocab)

21011

In [39]:
def encode(example):
    return {
        'input_ids':[vocab.get(token.lower(),1) for token in example['tokens']],
        'labels':example['ner_tags']
    }
        

In [40]:
dataset=dataset.map(encode)

Map: 100%|██████████| 3453/3453 [00:00<00:00, 11243.39 examples/s]


In [41]:
dataset['train'][0]

{'id': '0',
 'tokens': ['EU',
  'rejects',
  'German',
  'call',
  'to',
  'boycott',
  'British',
  'lamb',
  '.'],
 'pos_tags': [22, 42, 16, 21, 35, 37, 16, 21, 7],
 'chunk_tags': [11, 21, 11, 12, 21, 22, 11, 12, 0],
 'ner_tags': [3, 0, 7, 0, 0, 0, 7, 0, 0],
 'input_ids': [2, 3, 4, 5, 6, 7, 8, 9, 10],
 'labels': [3, 0, 7, 0, 0, 0, 7, 0, 0]}

In [43]:
def padding(batch):
    maxlen= max(len(example['input_ids']) for example in batch)
    input_ids=[]
    labels=[]
    mask=[]
    for example in batch:
        L=len(example['input_ids'])
        padlen=maxlen-L
        input_ids.append(example['input_ids']+[0]*padlen)
        labels.append(example['labels']+[0]*padlen)
        mask.append([1]*L+[0]*padlen)
    return torch.tensor(input_ids),torch.tensor(labels),torch.tensor(mask)

In [44]:
from torch.utils.data import DataLoader
train_loader=DataLoader(dataset['train'],batch_size=32,shuffle=True,collate_fn=padding)
test_loader=DataLoader(dataset['test'],batch_size=32,shuffle=True,collate_fn=padding)

In [45]:
#Scratch LSTM
#Emb--->LSTM--->output
import torch
import torch.nn as nn
class LSTMModel(nn.Module):
    def __init__(self,vocab_size,embedding_dim,hidden_dim,num_tags):
        super(LSTMModel,self).__init__()
        self.hidden_dim=hidden_dim
        #Embedding layer
        self.embedding=nn.Embedding(vocab_size,embedding_dim)
        #LSTM cell
        self.input_layer=nn.Linear(embedding_dim+hidden_dim,hidden_dim)
        self.output_layer=nn.Linear(embedding_dim+hidden_dim,hidden_dim)
        self.forget_layer=nn.Linear(embedding_dim+hidden_dim,hidden_dim)
        self.candidate_layer=nn.Linear(embedding_dim+hidden_dim,hidden_dim)
        #output_layer
        self.fc=nn.Linear(hidden_dim,num_tags)
    def forward(self,input_ids,mask):
        outputs=[]
        emb=self.embedding(input_ids)
        B,T,d=emb.shape
        h=torch.zeros(B,self.hidden_dim,device=input_ids.device)
        c=torch.zeros(B,self.hidden_dim,device=input_ids.device)
        for t in range(T):
            xt=emb[:,t,:]
            concatenated=torch.cat((xt,h),dim=1)
            i=torch.sigmoid(self.input_layer(concatenated))
            f=torch.sigmoid(self.forget_layer(concatenated))
            o=torch.sigmoid(self.output_layer(concatenated))
            cand=torch.tanh(self.candidate_layer(concatenated))
            c=i*cand+f*c
            h=o*torch.tanh(c)
            h=h*mask[:,t].unsqueeze(1)
            c=c*mask[:,t].unsqueeze(1)
            out=self.fc(h)
            outputs.append(out)
        return torch.stack(outputs,dim=1)
            

In [46]:
device=torch.device('cuda' if torch.cuda.is_available else 'cpu')
model=LSTMModel(len(vocab),128,256,len(tagnames)).to(device)

In [47]:
optimizer=torch.optim.Adam(model.parameters())

In [48]:
def masked_loss(outputs,labels,mask):
    B,T,c=outputs.shape
    outputs=outputs.view(B*T,c)
    labels=labels.view(B*T)
    mask=mask.view(B*T)
    loss=torch.nn.functional.cross_entropy(outputs,labels,reduction='none')
    loss=loss*mask
    return loss.sum()/mask.sum()

In [49]:
def masked_accuracy(outputs,labels,mask):
    pred_label=torch.argmax(outputs,dim=-1)
    correct=(pred_label==labels).float()
    correct=correct*mask
    return correct.sum()/mask.sum()

In [50]:
epochs=10
from tqdm import tqdm
for epoch in range(epochs):
    total_loss=0
    total_accuracy=0
    model.train()
    loop=tqdm(train_loader,desc=f'Epoch: {epoch+1}/{epochs}')
    for input_ids,labels,mask in loop:
        optimizer.zero_grad()
        input_ids,labels,mask=input_ids.to(device),labels.to(device),mask.to(device)
        y_pred=model(input_ids,mask)
        batch_loss=masked_loss(y_pred,labels,mask)
        batch_loss.backward()
        optimizer.step()
        total_loss+=batch_loss.item()
        batch_accuracy=masked_accuracy(y_pred,labels,mask).item()
        total_accuracy+=batch_accuracy
        loop.set_postfix(loss=batch_loss.item(),accuracy=batch_accuracy)
    print(f'Epoch:{epoch+1}|Loss:{total_loss/len(train_loader)}|Accuracy:{total_accuracy/len(train_loader)}')                                                             
                                                             
    

Epoch: 1/10: 100%|██████████| 439/439 [00:23<00:00, 18.75it/s, accuracy=0.881, loss=0.406]


Epoch:1|Loss:0.6031285019290202|Accuracy:0.8414814086686099


Epoch: 2/10: 100%|██████████| 439/439 [00:22<00:00, 19.14it/s, accuracy=0.943, loss=0.2]  


Epoch:2|Loss:0.3073820080056549|Accuracy:0.9087723177223379


Epoch: 3/10: 100%|██████████| 439/439 [00:23<00:00, 18.82it/s, accuracy=0.937, loss=0.236] 


...

Epoch:8|Loss:0.018311819777630932|Accuracy:0.9951675940211651


Epoch: 9/10: 100%|██████████| 439/439 [00:22<00:00, 19.72it/s, accuracy=0.993, loss=0.0173] 


Epoch:9|Loss:0.013702588225343607|Accuracy:0.9962318765003751


Epoch: 10/10: 100%|██████████| 439/439 [00:22<00:00, 19.39it/s, accuracy=0.993, loss=0.016]  

Epoch:10|Loss:0.011834849706365131|Accuracy:0.9965517819606633


In [56]:
class GRUModel(nn.Module):
    def __init__(self,vocab_size,embedding_dim,hidden_dim,num_tags):
        super(GRUModel,self).__init__()
        self.hidden_dim=hidden_dim
        self.embedding=nn.Embedding(vocab_size,embedding_dim)
        self.reset_gate=nn.Linear(embedding_dim+hidden_dim,hidden_dim)
        self.update_gate=nn.Linear(embedding_dim+hidden_dim,hidden_dim)
        self.candidate_gate=nn.Linear(embedding_dim+hidden_dim,hidden_dim)
        self.output_layer=nn.Linear(hidden_dim,num_tags)
    def forward(self,input_ids,mask):
        emb=self.embedding(input_ids)
        B,T,d=emb.shape
        h=torch.zeros(B,self.hidden_dim,device=input_ids.device)
        outputs=[]
        for t in range(T):
            xt=emb[:,t,:]
            concatenated=torch.cat((xt,h),dim=1)
            r=torch.sigmoid(self.reset_gate(concatenated))
            z=torch.sigmoid(self.update_gate(concatenated))
            rh=r*h
            xrh=torch.cat((xt,rh),dim=1)
            cand=torch.tanh(self.candidate_gate(xrh))
            h=z*cand+(1-z)*h
            out=self.output_layer(h)
            outputs.append(out)
        return torch.stack(outputs,dim=1)

In [57]:
device=torch.device('cuda' if torch.cuda.is_available else 'cpu')
model1=GRUModel(len(vocab),128,256,len(tagnames)).to(device)

In [58]:
optimizer=torch.optim.Adam(model1.parameters())

In [59]:
epochs=10
from tqdm import tqdm
for epoch in range(epochs):
    total_loss=0
    total_accuracy=0
    model.train()
    loop=tqdm(train_loader,desc=f'Epoch: {epoch+1}/{epochs}')
    for input_ids,labels,mask in loop:
        optimizer.zero_grad()
        input_ids,labels,mask=input_ids.to(device),labels.to(device),mask.to(device)
        y_pred=model1(input_ids,mask)
        batch_loss=masked_loss(y_pred,labels,mask)
        batch_loss.backward()
        optimizer.step()
        total_loss+=batch_loss.item()
        batch_accuracy=masked_accuracy(y_pred,labels,mask).item()
        total_accuracy+=batch_accuracy
        loop.set_postfix(loss=batch_loss.item(),accuracy=batch_accuracy)
    print(f'Epoch:{epoch+1}|Loss:{total_loss/len(train_loader)}|Accuracy:{total_accuracy/len(train_loader)}')                                                             
                                                             
    

Epoch: 1/10: 100%|██████████| 439/439 [00:19<00:00, 22.34it/s, accuracy=0.892, loss=0.367]


Epoch:1|Loss:0.5778111833103153|Accuracy:0.848307378700423


Epoch: 2/10: 100%|██████████| 439/439 [00:19<00:00, 22.21it/s, accuracy=0.923, loss=0.258]


Epoch:2|Loss:0.2988715186409635|Accuracy:0.9124258976590932


Epoch: 3/10: 100%|██████████| 439/439 [00:19<00:00, 22.19it/s, accuracy=0.965, loss=0.103] 


...

Epoch:8|Loss:0.017125713895658427|Accuracy:0.995278739440414


Epoch: 9/10: 100%|██████████| 439/439 [00:20<00:00, 21.85it/s, accuracy=0.997, loss=0.0222] 


Epoch:9|Loss:0.013918494235550251|Accuracy:0.9962232613889395


Epoch: 10/10: 100%|██████████| 439/439 [00:20<00:00, 21.57it/s, accuracy=0.997, loss=0.0121] 

Epoch:10|Loss:0.011854625998885883|Accuracy:0.9964748449912104


In [62]:
class LSTM_In_Built(nn.Module):
    def __init__(self,vocab_size,embedding_dim,hidden_dim,num_tags):
        super(LSTM_In_Built,self).__init__()
        self.embedding=nn.Embedding(vocab_size,embedding_dim)
        self.lstm=nn.LSTM(input_size=embedding_dim,hidden_size=hidden_dim,batch_first=True)
        self.fc=nn.Linear(hidden_dim,num_tags)
    def forward(self,input_dim,mask):
        emb=self.embedding(input_ids)
        lstm_output,_=self.lstm(emb)
        output=self.fc(lstm_output)
        return output
    

In [63]:
device=torch.device('cuda' if torch.cuda.is_available else 'cpu')
model2=LSTM_In_Built(len(vocab),128,256,len(tagnames)).to(device)

In [64]:
optimizer=torch.optim.Adam(model2.parameters())

In [65]:
epochs=10
from tqdm import tqdm
for epoch in range(epochs):
    total_loss=0
    total_accuracy=0
    model.train()
    loop=tqdm(train_loader,desc=f'Epoch: {epoch+1}/{epochs}')
    for input_ids,labels,mask in loop:
        optimizer.zero_grad()
        input_ids,labels,mask=input_ids.to(device),labels.to(device),mask.to(device)
        y_pred=model2(input_ids,mask)
        batch_loss=masked_loss(y_pred,labels,mask)
        batch_loss.backward()
        optimizer.step()
        total_loss+=batch_loss.item()
        batch_accuracy=masked_accuracy(y_pred,labels,mask).item()
        total_accuracy+=batch_accuracy
        loop.set_postfix(loss=batch_loss.item(),accuracy=batch_accuracy)
    print(f'Epoch:{epoch+1}|Loss:{total_loss/len(train_loader)}|Accuracy:{total_accuracy/len(train_loader)}')                                                             
                                                             
    

Epoch: 1/10: 100%|██████████| 439/439 [00:03<00:00, 122.91it/s, accuracy=0.88, loss=0.439] 


Epoch:1|Loss:0.5993042060753208|Accuracy:0.8461971728466366


Epoch: 2/10: 100%|██████████| 439/439 [00:03<00:00, 124.99it/s, accuracy=0.906, loss=0.273]


Epoch:2|Loss:0.29992608597039633|Accuracy:0.9114386823562934


Epoch: 3/10: 100%|██████████| 439/439 [00:03<00:00, 125.37it/s, accuracy=0.96, loss=0.111]  


...

Epoch:8|Loss:0.01801088225501664|Accuracy:0.9952491779805316


Epoch: 9/10: 100%|██████████| 439/439 [00:03<00:00, 127.72it/s, accuracy=1, loss=0.00748]    


Epoch:9|Loss:0.013860686211199723|Accuracy:0.996071489905442


Epoch: 10/10: 100%|██████████| 439/439 [00:03<00:00, 127.61it/s, accuracy=0.997, loss=0.00654]

Epoch:10|Loss:0.011370216279899476|Accuracy:0.9966375271117226
